<a href="https://colab.research.google.com/github/Abdul769469/MCP_SERVER/blob/main/YouTube_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-groq langchain-core \
            langchain-text-splitters faiss-cpu sentence-transformers \
            youtube-transcript-api -q

print("✅ All packages installed!")

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("✅ Groq API key loaded securely!")

In [ ]:
!pip install langchain-text-splitters -q

In [ ]:
# Run this to see exact available methods
from youtube_transcript_api import YouTubeTranscriptApi
methods = [m for m in dir(YouTubeTranscriptApi) if not m.startswith('_')]
print(methods)

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

cookie_file = "/content/cookies.txt"

def get_video_id(url):
    if "watch?v=" in url:
        return url.split("watch?v=")[-1].split("&")[0]
    elif "youtu.be/" in url:
        return url.split("youtu.be/")[-1].split("?")[0].split("&")[0]
    elif "shorts/" in url:
        return url.split("shorts/")[-1].split("?")[0]
    else:
        raise ValueError("❌ Invalid YouTube URL format!")

def fetch_transcript(youtube_url):
    video_id = get_video_id(youtube_url)
    print(f"🎬 Fetching transcript for video ID: {video_id}")

    # Method 1 — instance with cookies
    try:
        ytt = YouTubeTranscriptApi(cookies=cookie_file)
        fetched = ytt.fetch(video_id, languages=["en"])
        full_text = " ".join([snippet.text for snippet in fetched])
        print("✅ Got transcript with cookies!")
        return full_text
    except Exception as e:
        print(f"⚠️ Method 1 failed: {e}")

    # Method 2 — instance without cookies
    try:
        ytt = YouTubeTranscriptApi()
        fetched = ytt.fetch(video_id, languages=["en"])
        full_text = " ".join([snippet.text for snippet in fetched])
        print("✅ Got transcript without cookies!")
        return full_text
    except Exception as e:
        print(f"⚠️ Method 2 failed: {e}")

    # Method 3 — list all and pick first
    try:
        ytt = YouTubeTranscriptApi(cookies=cookie_file)
        transcript_list = ytt.list(video_id)
        for transcript in transcript_list:
            fetched = transcript.fetch()
            full_text = " ".join([snippet.text for snippet in fetched])
            print(f"✅ Got transcript in: {transcript.language}")
            return full_text
    except Exception as e:
        raise Exception(f"❌ All methods failed: {e}")

YOUTUBE_URL = "https://youtu.be/vz0OLOatU5c?si=vr_vTvMpQrAtyROG"

transcript_text = fetch_transcript(YOUTUBE_URL)
print(f"\n✅ Total characters: {len(transcript_text)}")
print(f"\n📝 Preview:\n{transcript_text[:500]}...")

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

doc = Document(
    page_content=transcript_text,
    metadata={"source": YOUTUBE_URL}
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunks = splitter.split_documents([doc])

print(f"✅ Transcript split into {len(chunks)} chunks")
print(f"\n📄 Sample chunk:\n{chunks[0].page_content[:300]}...")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("⏳ Loading embedding model, please wait...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embeddings)
print("✅ Vector store created!")

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

print("✅ Groq LLM ready!")

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

combined_text = " ".join([chunk.page_content for chunk in chunks])

if len(combined_text) > 12000:
    combined_text = combined_text[:12000]

prompt = PromptTemplate(
    template="""You are a helpful assistant. Summarize this YouTube video
transcript into a well-structured summary with:

1. 🎯 Main Topic (1-2 sentences)
2. 🔑 Key Points (bullet points)
3. 💡 Important Insights (bullet points)
4. 📌 Conclusion (1-2 sentences)

Transcript:
{text}

SUMMARY:""",
    input_variables=["text"]
)

chain = prompt | llm | StrOutputParser()

print("⏳ Generating summary, please wait...")
summary_text = chain.invoke({"text": combined_text})

print("\n" + "="*60)
print("📺 VIDEO SUMMARY")
print("="*60)
print(summary_text)
print("="*60)

summary_result = {"output_text": summary_text}

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

qa_prompt_template = """You are an assistant that answers questions about
a YouTube video based strictly on its transcript.
Only use the context below to answer. If the answer is not in the
transcript, say "That wasn't covered in this video."

Context from transcript:
{context}

Question: {question}

Answer:"""

QA_PROMPT = PromptTemplate(
    template=qa_prompt_template,
    input_variables=["context", "question"]
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | QA_PROMPT
    | llm
    | StrOutputParser()
)

print("✅ RAG Q&A chain ready!")

✅ RAG Q&A chain ready!


In [ ]:
def ask(question):
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    answer = qa_chain.invoke(question)
    print(f"\n🤖 Answer:\n{answer}")
    print("="*60)

# ✏️ Change these to match your video
ask("What is the main topic of this video?")
ask("What are the most important points discussed?")
ask("What conclusions or recommendations were made?")

In [ ]:
print("💬 Chat with the YouTube Video!")
print("   Ask anything about the video content.")
print("   Type 'summary' to reprint the summary.")
print("   Type 'quit' to exit.\n")

while True:
    question = input("You: ").strip()

    if not question:
        continue

    if question.lower() in ["quit", "exit", "q"]:
        print("👋 Goodbye!")
        break

    if question.lower() == "summary":
        print(f"\n📝 Summary:\n{summary_result['output_text']}\n")
        continue

    answer = qa_chain.invoke(question)
    print(f"\n🤖 Bot: {answer}\n")